# 02 - VAR Modeling and Diagnostics

Notebook goals:
- Compare lag-order suggestions (AIC/BIC/HQIC/FPE)
- Fit baseline VAR on transformed monthly data
- Check stability via inverse roots and unit circle
- Check residual whiteness/autocorrelation

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.tsa.api import VAR

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from nasdaq_svar.config import load_config
from nasdaq_svar.pipeline import run_pipeline
from nasdaq_svar.presentation import build_chapter2_conclusion, whiteness_pvalue

plt.style.use("seaborn-v0_8-whitegrid")
cfg = load_config(PROJECT_ROOT / "configs/default.yaml")

In [ ]:
stage2_path = PROJECT_ROOT / "data/processed/stage2_transformed_dataset.csv"
if not stage2_path.exists():
    _ = run_pipeline(config_path=PROJECT_ROOT / "configs/default.yaml", project_root=PROJECT_ROOT)

stage2_data = pd.read_csv(stage2_path, index_col=0, parse_dates=True)
stage2_data.index = pd.PeriodIndex(stage2_data.index, freq="M")
stage2_data.head()

In [ ]:
var_model = VAR(stage2_data)
lag_order = var_model.select_order(maxlags=int(cfg["stage2"]["maxlags"]))
lag_order_table = pd.DataFrame([lag_order.selected_orders])
lag_order_table

In [ ]:
selected_lag = lag_order.selected_orders.get(cfg["stage2"]["ic"], cfg["stage2"]["fallback_lag"])
selected_lag = max(int(selected_lag), 1)
var_fit = var_model.fit(selected_lag)
print(f"Selected lag ({cfg['stage2']['ic']}):", selected_lag)
print(var_fit.summary())

In [ ]:
roots = var_fit.roots
inv_roots = 1.0 / roots

theta = np.linspace(0, 2 * np.pi, 400)
unit_x = np.cos(theta)
unit_y = np.sin(theta)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(unit_x, unit_y, "k--", lw=1.0, label="Unit circle")
ax.scatter(inv_roots.real, inv_roots.imag, color="#1f77b4", s=40)
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(0, color="gray", lw=0.8)
ax.set_title("Inverse Roots of VAR Characteristic Polynomial")
ax.set_xlabel("Real")
ax.set_ylabel("Imaginary")
ax.set_aspect("equal", adjustable="box")
ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
whiteness = var_fit.test_whiteness(nlags=12)
print(whiteness.summary())

## Slide-Ready Conclusion

In [ ]:
chapter2_text = build_chapter2_conclusion(
    selected_lag=selected_lag,
    lag_order_table=lag_order_table,
    inv_roots=inv_roots,
    whiteness_pvalue=whiteness_pvalue(whiteness),
    chosen_ic=cfg["stage2"]["ic"],
)
display(Markdown(chapter2_text))